# Theme analysis Plots

We will do our plots with Flourish. We just need to create the csv and export them in their online plateform to obtain our plots.

In [1]:
import os
from pathlib import Path

import pandas as pd

# statistical test
from scipy import stats

from IPython.display import HTML

# Custom imports
import utils.utils_themes_P3 as p3_util

In [4]:
# General variables: paths
ROOT = Path("../../data/newyorker_caption_contest")
THEME_PATH = ROOT / "THEMES_RESULTS_V4" 

# Files paths
DOCS_PATH   = THEME_PATH / "docs_with_themes.csv"
TF_IDF_PATH = THEME_PATH / "TF_IDF_themes_scores.csv"
THEMES_PATH = THEME_PATH / "themes.csv"

print(f"THEME_PATH:  {THEME_PATH.resolve()}\n")
print(f"DOCS_PATH:   {DOCS_PATH.resolve()}")
print(f"TF_IDF_PATH: {TF_IDF_PATH.resolve()}")
print(f"THEMES_PATH: {THEMES_PATH.resolve()}")

THEME_PATH:  /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4

DOCS_PATH:   /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4/docs_with_themes.csv
TF_IDF_PATH: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4/TF_IDF_themes_scores.csv
THEMES_PATH: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4/themes.csv


In [5]:
def load_csv(csv_path: Path, expected_cols) -> pd.DataFrame:
    """Load a csv file into Pandas DataFrame and check for expected columns."""
    
    df = pd.read_csv(csv_path)
    
    # check columns
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing expected columns: {missing}")
    
    return df

In [6]:
# load all content with themes extraction csv
df_doc = p3_util.load_docs_with_theme_csv(DOCS_PATH)
df_doc.head()

,doc_id,contest_id,kind,text,clean_text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,theme_id,theme_label
0,image_510,510,image,A man is relaxing on a city street. Others are...,man relax city street go business step pass ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,Train
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,congressman obstruction job,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,64,Election
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",mean middle class step,1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,53,Market
3,caption_510_2,510,caption,Does this suit make me look flat?,suit look flat,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,76,Clothing
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",right woman come know,3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,45,Marriage


In [7]:
# load generated label dictionary csv
gen_labels_cols = ["caption_id", "gen_label"]
df_tfidf = load_csv(GEN_LABELS_PATH, expected_cols=gen_labels_cols)

# rename 'caption_id' into 'theme_id' for clarity
df_tfidf = df_tfidf.rename(columns={"caption_id": "theme_id"})

# add generated labels to df_doc by matching on 'theme_id'
df_doc = df_doc.merge(df_tfidf, on="theme_id", how="left")

df_doc.head()

,doc_id,contest_id,kind,text,clean_text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,theme_id,theme_label,gen_label
0,image_510,510,image,A man is relaxing on a city street. Others are...,man relax city street go business step pass ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,Train,transport
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,congressman obstruction job,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,64,Election,Trump
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",mean middle class step,1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,53,Market,commerce
3,caption_510_2,510,caption,Does this suit make me look flat?,suit look flat,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,76,Clothing,apparel
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",right woman come know,3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,45,Marriage,marriage


In [9]:
# load similarity theme_id matrix
similarity_df = pd.read_csv(SIMILARITY_PATH)
similarity_df.head()

,theme_id,0,1,2,3,4,5,6,7,8,...,70,71,72,73,74,75,76,77,78,79
0,0,1.000000,0.402895,0.283682,0.362918,0.513067,0.381242,0.306716,0.410318,0.514566,...,0.390853,0.302836,0.346689,0.398186,0.434105,0.443288,0.352182,0.412290,0.409759,0.365864
1,1,0.402895,1.000000,0.327793,0.487851,0.510671,0.472350,0.286405,0.509334,0.582684,...,0.435116,0.420763,0.521627,0.557530,0.479820,0.442083,0.394563,0.537888,0.484329,0.518927
2,2,0.283682,0.327793,1.000000,0.334920,0.412829,0.288990,0.198154,0.415377,0.506217,...,0.373281,0.292525,0.543920,0.438732,0.419888,0.279726,0.359059,0.407414,0.328195,0.326667
3,3,0.362918,0.487851,0.334920,1.000000,0.462116,0.384542,0.223693,0.402566,0.454148,...,0.381653,0.299965,0.390584,0.384954,0.444621,0.408297,0.362750,0.436603,0.327778,0.328734
4,4,0.513067,0.510671,0.412829,0.462116,1.000001,0.525934,0.282231,0.605301,0.702932,...,0.561148,0.423270,0.488898,0.601121,0.545249,0.467601,0.494168,0.558372,0.527386,0.477060


In [74]:
# # load TF-IDF theme terms scores csv
# tfidf_cols = ["theme_id", "term", "sim", "local_freq", "global_freq", "distinctiveness", "score"]
# df_tfidf = load_csv(TF_IDF_PATH, expected_cols=tfidf_cols)
# df_tfidf.head()

In [ ]:
# # load the 80 themes id, terms and stats
# themes_cols = ["theme_id", "theme_label", "nb_instances", "mu_rank", "mu_mean", "mu_precision",
#                "mu_not_funny", "mu_somewhat_funny", "mu_funny", "TF_IDF_terms"]
# df_themes = load_csv(THEMES_PATH, expected_cols=themes_cols)
# df_themes.head()

## Theme relations between images and captions

Sketch:

<img src="../../figures/sketches/graph_sketch.jpg" width="400">

In [ ]:
df_flourish = p3_util.build_flourish_csv(df_doc)
df_flourish.head()


,Source,Target,value,source_id,target_id
0,T0: Dog,T0: Dog,5109,0,0
1,T0: Dog,T1: Fall Asleep,122,0,1
2,T0: Dog,T2: Year,352,0,2
3,T0: Dog,T3: Work Home,509,0,3
4,T0: Dog,T4: Clown,278,0,4


In [8]:
print(Path("../../data/newyorker_caption_contest/FLOURISH/flourish_alluvial.csv").resolve())

/Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/FLOURISH/flourish_alluvial.csv


In [9]:
df_flourish.to_csv("../../data/newyorker_caption_contest/FLOURISH/flourish_heatmap.csv", index=False)

In [ ]:
# Size reduction for readability

TOP_THEMES_PER_SIDE = 15   # keep top N nodes per side; rest -> removed
TOP_LINKS_PER_SOURCE = 5   # keep top K outgoing links per Source
MIN_VALUE = 1500             # drop links below this after aggregation
DROP_SELF_LINKS = False    # Source!=Target only if True

df_f = df_flourish.copy()

# optional: remove self-links
if DROP_SELF_LINKS:
    df_f = df_f[df_f["Source"] != df_f["Target"]].copy()

# keep only top nodes on each side (by total flow)
top_sources = (df_f.groupby("Source")["value"].sum().nlargest(TOP_THEMES_PER_SIDE).index)
top_targets = (df_f.groupby("Target")["value"].sum().nlargest(TOP_THEMES_PER_SIDE).index)

df_f = df_f[df_f["Source"].isin(top_sources) & df_f["Target"].isin(top_targets)].copy()

# drop tiny links
df_f = df_f[df_f["value"] >= MIN_VALUE].copy()

# keep only top K links per Source (within the retained nodes)
df_f = (df_f.sort_values(["Source", "value"], ascending=[True, False])
           .groupby("Source", as_index=False)
           .head(TOP_LINKS_PER_SOURCE)
           .reset_index(drop=True))

df_flourish_reduced = df_f

df_flourish_reduced.head()



In [ ]:
print("Length df_flourish:", len(df_flourish))
print("Length df_flourish_reduced:", len(df_flourish_reduced))

In [ ]:
df_flourish_reduced.to_csv("../../data/newyorker_caption_contest/FLOURISH/flourish_alluvial_reduced3.csv", index=False)

Results:

In [ ]:
# Display the Flourish Sankey diagram (with fixed width)
HTML("""
<div class="flourish-embed flourish-sankey" data-src="visualisation/26895805">
  <script src="https://public.flourish.studio/resources/embed.js"></script>
  <noscript>
    <img src="https://public.flourish.studio/visualisation/26895805/thumbnail" width="100%" alt="sankey visualization" />
  </noscript>
</div>
""")

In [ ]:
# Plot heatmap from full data for COUNT
fig = p3_util.heatmap_from_df(df_heatmap,
                      x_col="caption_theme_id",
                      y_col="image_theme_id",
                      value_col="count",
                      colors = ["#ffffff",
                                "#2c3e50",
                                "#2980b9",
                                "#ffd200",
                                "#e74c3c"],
                      title = "Heatmap (all data)",
                      width=800)

fig.show()

In [ ]:
# Plot heatmap from full data for COUNT
fig = p3_util.heatmap_from_df(df_heatmap,
                      x_col="caption_theme_id",
                      y_col="image_theme_id",
                      value_col="funny_score",
                      colors = ["#ffffff",
                                "#2c3e50",
                                "#2980b9",
                                "#ffd200",
                                "#e74c3c"],
                      title = "Heatmap (all data)",
                      width=800)

fig.show()

In [ ]:
fig = p3_util.heatmap_from_df(df_heatmap_3,
                      x_col="caption_theme_id",
                      y_col="image_theme_id",
                      value_col="count",
                      colors = ["#ffffff",
                                "#4B5D6E",
                                "#2c3e50",
                                "#2980b9",
                                "#ffd200",
                                "#e74c3c"],
                      title = "Heatmap (all data)",
                      width=5000)

fig.show()

In [88]:
# colors=["#ffffff",
#         "#4B5D6E",
#         "#2c3e50",
#         "#2980b9",
#         "#6fa1d3", 
#         "#27ae60",
#         "#ffd200",
#         "#e74c3c"]

# colors = ["#ffffff",
#         "#4B5D6E",
#         "#2c3e50",
#         "#ffd200",
#         "#6fa1d3",
#         "#0f4a85",
#         "#27ae60",
#         "#e74c3c"]

In [86]:
# # Welch t-test
# stats_out = {}
# if np.issubdtype(df[x_col].dtype, np.number) and np.issubdtype(df[y_col].dtype, np.number):
#     r, p = stats.pearsonr(df[x_col], df[y_col])
#     stats_out["correlation"] = {"r": r, "p_value": p, "significant": p < alpha}

# theme_tests = []
# n_tests = df[image_theme_col].nunique()
# adj_alpha = alpha / max(n_tests, 1)
# for theme, vals in df.groupby(image_theme_col)[value_col]:
#     others = df.loc[df[image_theme_col] != theme, value_col]
#     if len(vals) < 2 or len(others) < 2:
#         continue
#     t_stat, p_val = stats.ttest_ind(vals, others, equal_var=False)

#     mean_other = others.mean()
#     var1, var2 = vals.var(ddof=1), others.var(ddof=1)
#     n1, n2 = len(vals), len(others)
#     se = np.sqrt(var1 / n1 + var2 / n2)

#     df_welch = (var1 / n1 + var2 / n2) ** 2 / (
#         (var1**2) / (n1**2 * (n1 - 1)) + (var2**2) / (n2**2 * (n2 - 1))
#     )
#     t_crit = stats.t.ppf(1 - adj_alpha / 2, df=df_welch)

#     threshold_high = mean_other + t_crit * se
#     threshold_low = mean_other - t_crit * se

#     theme_tests.append(
#         {
#             "theme": theme,
#             "mean": vals.mean(),
#             "p_value": p_val,
#             "significant": p_val < adj_alpha,
#             "threshold_low": threshold_low,
#             "threshold_high": threshold_high,
#         }
#     )

# stats_out["theme_tests"] = {
#     "alpha": alpha,
#     "adjusted_alpha": adj_alpha,
#     "results": theme_tests,
# }

Theme similarity plot

In [ ]:
# Plot theme similarity heatmap
fig = p3_util.theme_id_heatmap(similarity_df, width=600)
fig.show()

# Statistical tests

We are interested at studying how **image themes** and **caption themes** relate to:
- **frequency** (`count`)
- **caption funniness** (`funny_score` == `mean` column from origianl dataset).

Let us define again the questions we aim at answering:
1) **Same-theme frequency vs funniness:** For pairs where `image_theme_id == caption_theme_id`, are more frequent (higher `count`) connections also funnier (higher `funny_score`)?
2) **Inherent image-theme funniness:** Are some **image themes** associated with higher caption funniness on average (and if so, which themes)?
3) **Semantic distance effects:** Are **semantically distant** image–caption theme pairs (lower cosine similarity / higher distance) (a) funnier and/or (b) less/more frequent than semantically similar pairs?


In [11]:
doc_matrix_df = p3_util.build_theme_id_matrix(df_doc, matrix_value="count")

doc_matrix_df.head()

caption_theme_id,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
image_theme_id,,,,,,,,,,,,,,,,,,,,,
0,5109,122,352,509,278,85,29,224,2395,217,...,673,113,237,123,602,784,1550,574,43,88
1,7,1014,7,663,25,55,2,19,74,19,...,16,11,32,13,58,39,138,39,0,57
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,49,232,867,64,49,2,99,279,9,...,158,69,48,23,119,109,590,522,5,19
4,2629,478,824,1026,6015,314,111,459,2384,1390,...,627,1415,731,883,903,1407,597,942,141,1739
